# Debugging

This notebook debugs classifiers in the Multi-Output prediction system. 

In [77]:
# Automatically reload imported modules before executing code
%load_ext autoreload
%autoreload

In [78]:
from nbutils import setup_path
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

setup_path()

## Debugging

On the training dataset, the Motown tag had a too low threshold, and we got a lot of false positives. 
On the inference dataset, the Eurodance tag had a too low threshold

- Motown: Threshold sits at 0, so loads of false positives.
    - Should be set on 0.4
- Eurodance, threshold quite low. Loads of false positives on inference. 
    - Should be set on 0.9
- Ambient: Loads of false positives as well. Classifier not picking up correctly. 

In [151]:
import polars as pl

tags_df = pl.read_csv("../data/unique_tags.csv")

In [152]:
from nbutils import  display_polars

display_polars(tags_df)

shape: (50, 4)
┌────────────┬──────────────────────────────────────┬────────────────┬───────────────────────┐
│ TagID      ┆ UUID                                 ┆ TagGroup       ┆ TagName               │
│ ---        ┆ ---                                  ┆ ---            ┆ ---                   │
│ i64        ┆ str                                  ┆ str            ┆ str                   │
╞════════════╪══════════════════════════════════════╪════════════════╪═══════════════════════╡
│ 1429694612 ┆ 3e973c3c-001b-4792-92f9-684d40943024 ┆ Genre          ┆ Techno                │
│ 2484825285 ┆ 22a59e2a-b3a6-470e-ad4b-92eddadcea7b ┆ Genre          ┆ Ambient               │
│ 3917148722 ┆ 7226a04c-8d3e-4f5a-b6be-acb6c5418df3 ┆ Genre          ┆ Hip-Hop               │
│ 323305339  ┆ e8484566-7872-449c-bbb6-e9002b335b49 ┆ Genre          ┆ Motown                │
│ 385085509  ┆ c9ab7b83-30a5-4797-b70c-7f18d7d553d8 ┆ Genre          ┆ Funk                  │
│ 3244404646 ┆ c4689a95-ddbf-4c07-9

In [153]:
# Config
LABEL_TO_DEBUG = "Techno" # Motown, Eurodance, Ambient
SHOW_TRAINING = False

In [154]:
import polars as pl
from models import load_model

# Load model (see hyper parmam tuning)
model_data = load_model("../models/xgboost_Genre_20251015_232027_model.pkl")

label_idx = model_data['tags'].index(LABEL_TO_DEBUG)

# Check current threshold
current_threshold = model_data['thresholds'][label_idx]
print(f"Current threshold for '{LABEL_TO_DEBUG}': {current_threshold:.4f}")

# If you have validation metrics stored, check them
if 'per_label_metrics' in model_data:
    metrics = model_data['per_label_metrics']
    label_metrics = metrics.filter(pl.col("tag") == LABEL_TO_DEBUG)
    print(f"\nCurrent metrics:\n{label_metrics}")


MODEL LOADED SUCCESSFULLY
Model name: xgboost_Genre
Tag group: Genre
Number of labels: 22
Saved on: 2025-10-15T23:20:27.538256

Loaded components:
  Model: ✓
  Thresholds: ✓
  Scaler: ✓
  PCA: ✗

Stored metrics:
  macro_f1: 0.5060
  macro_precision: 0.4732
  macro_recall: 0.6371

Current threshold for 'Techno': 0.6388


## Check Current Threshold and Performance

In [155]:
from processing import predict_with_optimized_thresholds
from utils import get_clean_songs
from pyrekordbox import Rekordbox6Database

# Get songs and features
db = Rekordbox6Database()
songs_df = get_clean_songs(db, rename=True)
train_df = pl.read_parquet("../data/song_features.parquet")
inference_df = pl.read_parquet("../data/song_inference.parquet")

exclude_cols = [
    "song_path", "harmonic_percussive_ratio", "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]
feature_cols = [col for col in train_df.columns if col not in exclude_cols + ["song_id", "song_path"]]


# Get scaled features (same as training)
X_train = train_df.select(["song_id"] + feature_cols)
X_train_scaled = model_data['scaler'].transform(X_train.drop("song_id"))
X_train_scaled = pl.concat([X_train.select("song_id"), X_train_scaled], how="horizontal")

X_inference = inference_df.select(["song_id"] + feature_cols)
X_inference_scaled = model_data['scaler'].transform(X_inference.drop("song_id"))
X_inference_scaled = pl.concat([X_inference.select("song_id"), X_inference_scaled], how="horizontal")

# Get predictions with current thresholds
predictions_train = predict_with_optimized_thresholds(
    models=model_data['model'],
    X_test=X_train_scaled,
    tags=model_data['tags'],
    thresholds=model_data['thresholds'],
    songs_df=songs_df 
)

predictions_inference = predict_with_optimized_thresholds(
    models=model_data['model'],
    X_test=X_inference_scaled,
    tags=model_data['tags'],
    thresholds=model_data['thresholds'],
    songs_df=songs_df
)

[09:34:09] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


First, let's check how many of `LABEL_TO_DEBUG` are predicted. 

In [156]:
from nbutils import display_polars

base_scaled_df = X_train_scaled if SHOW_TRAINING else X_inference_scaled 
base_predictions_df =  predictions_train if SHOW_TRAINING else predictions_inference

# Find songs predicted as Motown
label_prediction = base_predictions_df.filter(
    pl.col("predicted_tags").list.contains(LABEL_TO_DEBUG)
)

print(f"\nSongs predicted as '{LABEL_TO_DEBUG}': {len(label_prediction)}")
display_polars(label_prediction.select(["song_title", "artist_name", "predicted_tags"]))


Songs predicted as 'Techno': 84
shape: (50, 3)
┌──────────────────────────────────┬─────────────────────────────┬─────────────────────────────────┐
│ song_title                       ┆ artist_name                 ┆ predicted_tags                  │
│ ---                              ┆ ---                         ┆ ---                             │
│ str                              ┆ str                         ┆ list[str]                       │
╞══════════════════════════════════╪═════════════════════════════╪═════════════════════════════════╡
│ Lefty's bar (Brame & Hamo Remix) ┆ Fouk                        ┆ ["Ambient", "House", "Motown",  │
│                                  ┆                             ┆ "Soul", "Techno", "Trance",     │
│                                  ┆                             ┆ "Tribal"]                       │
│ Fusion                           ┆ Janaret                     ┆ ["Ambient", "House", "Motown",  │
│                                  ┆       

## Check Prediction Probabilities

In [160]:
from models import predict_with_threshold

# Get probabilities for all predictions
_, probs = predict_with_threshold(
    model_data['model'],
    base_scaled_df.drop("song_id"),
    threshold=0.5
)

# Get probabilities for our label
label_probs = probs[:, label_idx]

# Create a dataframe with song_id and probabilities
probs_df = pl.DataFrame({
    "song_id": base_scaled_df["song_id"],
    f"{LABEL_TO_DEBUG}_probability": label_probs
})

# Join probabilities with predictions
label_prob_df = (
    base_predictions_df
    .join(probs_df, on="song_id", how="left")
    .filter(pl.col("predicted_tags").list.contains(LABEL_TO_DEBUG))
    .select(["song_title", "artist_name", f"{LABEL_TO_DEBUG}_probability", "predicted_tags"])
    .sort(f"{LABEL_TO_DEBUG}_probability", descending=True)
)

print(f"\nSongs predicted as '{LABEL_TO_DEBUG}' with probabilities:")
print(label_prob_df)

# Find the probability range
print(f"\nProbability range for '{LABEL_TO_DEBUG}' predictions:")
print(f"  Min: {label_prob_df[f'{LABEL_TO_DEBUG}_probability'].min():.4f}")
print(f"  Max: {label_prob_df[f'{LABEL_TO_DEBUG}_probability'].max():.4f}")
print(f"  Mean: {label_prob_df[f'{LABEL_TO_DEBUG}_probability'].mean():.4f}")
print(f"  Current threshold: {current_threshold:.4f}")


Songs predicted as 'Techno' with probabilities:
shape: (84, 4)
┌─────────────────────────┬─────────────────────────┬────────────────────┬─────────────────────────┐
│ song_title              ┆ artist_name             ┆ Techno_probability ┆ predicted_tags          │
│ ---                     ┆ ---                     ┆ ---                ┆ ---                     │
│ str                     ┆ str                     ┆ f32                ┆ list[str]               │
╞═════════════════════════╪═════════════════════════╪════════════════════╪═════════════════════════╡
│ Gimme The Funk          ┆ D-SKO (Djebali &        ┆ 0.97665            ┆ ["Ambient", "Bass",     │
│ [PODD003]               ┆ Rossko)                 ┆                    ┆ "Breakbeat", "Garage",  │
│                         ┆                         ┆                    ┆ "House", "Motown",      │
│                         ┆                         ┆                    ┆ "Techno", "Trance"]     │
│ Auto Machine (Original  ┆

In [162]:
# Look at the probability distribution
# If many false positives are in range 0.25-0.40 and current threshold is 0.25,
# try increasing to 0.35 or 0.40

test_thresholds = [0.3, 0.35, 0.4, 0.45, 0.50, 0.7, 0.9]

for test_threshold in test_thresholds:
    # Apply test threshold
    test_thresholds_array = model_data['thresholds'].copy()
    test_thresholds_array[label_idx] = test_threshold

    # Make predictions
    test_preds = predict_with_optimized_thresholds(
        models=model_data['model'],
        X_test=base_scaled_df,
        tags=model_data['tags'],
        thresholds=test_thresholds_array,
        songs_df=songs_df
    )

    # Count predictions
    n_predictions = len(
        test_preds.filter(pl.col("predicted_tags").list.contains(LABEL_TO_DEBUG))
    )

    print(f"Threshold {test_threshold:.2f}: {n_predictions} predictions")


Threshold 0.30: 180 predictions
Threshold 0.35: 162 predictions
Threshold 0.40: 145 predictions
Threshold 0.45: 127 predictions
Threshold 0.50: 116 predictions
Threshold 0.70: 70 predictions
Threshold 0.90: 28 predictions


## Explore Results of Thresholds

Eurodance is not performing well 

In [164]:
from nbutils import display_polars

CUSTOM_THRESH = 0.8 # The threshold at which the label is applied


# Set threshold for label motown 
test_thresholds_array = model_data['thresholds'].copy()
test_thresholds_array[label_idx] = CUSTOM_THRESH


disp_df = (
    predict_with_optimized_thresholds(
        models=model_data['model'],
        X_test=base_scaled_df,
        tags=model_data['tags'],
        thresholds=test_thresholds_array,
        songs_df=songs_df
    )
    .filter(pl.col("predicted_tags").list.contains(LABEL_TO_DEBUG))
    .join(probs_df, on="song_id", how="left") # show probability of label
    .select("song_title", "artist_name", "tag_names", "predicted_tags", f"{LABEL_TO_DEBUG}_probability")
    .with_columns(pl.col("tag_names").list.join(", ").alias("actual_tags"))
    .with_columns(pl.col("predicted_tags").list.join(", ").alias("pred_tags"))
    .with_columns(
        # Intersection: elements in both lists
        pl.col("predicted_tags").list.set_intersection("tag_names").list.len().alias("intersection"),
        # Union: unique elements from both lists
        pl.col("predicted_tags").list.set_union("tag_names").list.len().alias("union")
    )
    .with_columns(
        (pl.col("intersection") / pl.col("union")).fill_null(1.0).alias("agreement")
    )
    .drop("tag_names", "predicted_tags")
)


display_polars(disp_df, disp_df.shape[0])

shape: (49, 8)
┌─────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────┬───────────┐
│ song_title  ┆ artist_nam ┆ Techno_pro ┆ actual_tag ┆ pred_tags  ┆ intersecti ┆ union ┆ agreement │
│ ---         ┆ e          ┆ bability   ┆ s          ┆ ---        ┆ on         ┆ ---   ┆ ---       │
│ str         ┆ ---        ┆ ---        ┆ ---        ┆ str        ┆ ---        ┆ u32   ┆ f64       │
│             ┆ str        ┆ f32        ┆ str        ┆            ┆ u32        ┆       ┆           │
╞═════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════╪═══════════╡
│ Lefty's bar ┆ Fouk       ┆ 0.89197    ┆ AUTOTAG    ┆ Ambient,   ┆ 0          ┆ 8     ┆ 0.0       │
│ (Brame &    ┆            ┆            ┆            ┆ House,     ┆            ┆       ┆           │
│ Hamo Remix) ┆            ┆            ┆            ┆ Motown,    ┆            ┆       ┆           │
│             ┆            ┆            ┆            ┆ Soul,      ┆         